# Exploratory Data Analysis (EDA): Train & Test Dataset
### Kaggle Machine Learning Competition - House Prices: Advanced Regression Techniques

Notebook ini berisi analisis data eksplorasi (**Exploratory Data Analysis / EDA**) secara komprehensif terhadap dataset **train.csv** dan **test.csv**.

---

### Tujuan EDA:
1. **Memahami Struktur & Dimensi Data**: Memeriksa ukuran baris dan kolom dari kedua dataset.
2. **Inspeksi Tipe Data**: Memeriksa kolom numerik dan kategorikal.
3. **Analisis Missing Values**: Mengidentifikasi kolom dengan data kosong pada data train maupun test.
4. **Analisis Variabel Target (`SalePrice`)**: Memeriksa distribusi, skewness, dan statistik ringkas harga rumah.
5. **Korelasi Antar Fitur**: Menemukan fitur-fitur yang memiliki hubungan linier terkuat dengan harga jual rumah.
6. **Perbandingan Fitur Terpilih (Train vs Test)**: Memastikan distribusi fitur pada data train dan data test konsisten.
7. **Deteksi Pencilan (Outlier)**: Memeriksa anomali data yang dapat memengaruhi performa model.


## 1. Import Library
Mengimpor pustaka yang dibutuhkan untuk pengolahan data (`pandas`, `numpy`) serta visualisasi data (`matplotlib`, `seaborn`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11


## 2. Memuat Dataset `train.csv` dan `test.csv`
Data dimuat menggunakan `pandas.read_csv()`. Disediakan fallback otomatis agar file dapat dibuka baik dari direktori lokal maupun path absolut.


In [ ]:
# Path dataset train dan test
train_path = "train.csv" if os.path.exists("train.csv") else r"D:\file baru\mendata\machine learning competition\train.csv"
test_path = "test.csv" if os.path.exists("test.csv") else r"D:\file baru\mendata\machine learning competition\test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Data Train berhasil dimuat: {df_train.shape[0]} baris, {df_train.shape[1]} kolom")
print(f"Data Test berhasil dimuat : {df_test.shape[0]} baris, {df_test.shape[1]} kolom")


## 3. Pratinjau Data (Head & Dimensi)
Melihat 5 baris pertama dari data train dan test untuk mengenali gambaran fitur-fitur yang ada.


In [ ]:
# 5 Baris pertama data train
print("=== 5 BARIS PERTAMA DATA TRAIN ===")
display(df_train.head())


In [ ]:
# 5 Baris pertama data test
print("=== 5 BARIS PERTAMA DATA TEST ===")
display(df_test.head())


Perhatikan bahwa data train memiliki 81 kolom (termasuk kolom target `SalePrice`), sedangkan data test memiliki 80 kolom (tidak memiliki kolom `SalePrice` karena itulah yang akan diprediksi).


In [ ]:
# Memeriksa selisih kolom antara train dan test
kolom_hanya_di_train = set(df_train.columns) - set(df_test.columns)
print("Kolom yang ada di train tetapi tidak ada di test:", kolom_hanya_di_train)


## 4. Tipe Data dan Ringkasan Statistik
Melihat ringkasan tipe data (numerik vs kategorikal) serta statistik deskriptif dasar.


In [ ]:
print("=== RINGKASAN TIPE DATA TRAIN ===")
print(df_train.dtypes.value_counts())
print("\n=== RINGKASAN TIPE DATA TEST ===")
print(df_test.dtypes.value_counts())


In [ ]:
# Statistik deskriptif data train
print("=== STATISTIK DESKRIPTIF DATA TRAIN (NUMERIK) ===")
display(df_train.describe().round(2))


## 5. Analisis Missing Values (Nilai Hilang)
Pemeriksaan nilai `NaN` / kosong pada train dan test sangat penting untuk menentukan strategi pembersihan atau imputasi data sebelum pemodelan.


In [ ]:
# Menghitung missing values pada Train
null_train = df_train.isnull().sum()
null_train = null_train[null_train > 0].sort_values(ascending=False)
df_null_train = pd.DataFrame({
    'Jumlah_Null': null_train,
    'Persentase_Null (%)': (null_train / len(df_train)) * 100
})

# Menghitung missing values pada Test
null_test = df_test.isnull().sum()
null_test = null_test[null_test > 0].sort_values(ascending=False)
df_null_test = pd.DataFrame({
    'Jumlah_Null': null_test,
    'Persentase_Null (%)': (null_test / len(df_test)) * 100
})

print(f"Jumlah kolom yang memiliki missing value di Train: {len(df_null_train)}")
print(f"Jumlah kolom yang memiliki missing value di Test : {len(df_null_test)}")


In [ ]:
# Visualisasi 15 Kolom dengan Missing Value Terbanyak di Train vs Test
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=df_null_train['Persentase_Null (%)'].head(15), 
            y=df_null_train.head(15).index, 
            ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 15 Missing Values di Data Train (%)")
axes[0].set_xlabel("Persentase (%)")

sns.barplot(x=df_null_test['Persentase_Null (%)'].head(15), 
            y=df_null_test.head(15).index, 
            ax=axes[1], palette="Oranges_r")
axes[1].set_title("Top 15 Missing Values di Data Test (%)")
axes[1].set_xlabel("Persentase (%)")

plt.tight_layout()
plt.show()


## 6. Analisis Distribusi Variabel Target (`SalePrice`)
Melihat sebaran harga jual rumah pada data latih. Apakah terdistribusi normal atau mengalami *skewness* (kemiringan)?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram + KDE
sns.histplot(df_train['SalePrice'], kde=True, color='#1f77b4', ax=axes[0])
axes[0].set_title("Distribusi Harga Rumah (SalePrice)")
axes[0].set_xlabel("SalePrice ($)")
axes[0].axvline(df_train['SalePrice'].mean(), color='red', linestyle='--', label=f"Mean: ${df_train['SalePrice'].mean():,.0f}")
axes[0].axvline(df_train['SalePrice'].median(), color='green', linestyle='-', label=f"Median: ${df_train['SalePrice'].median():,.0f}")
axes[0].legend()

# Boxplot
sns.boxplot(x=df_train['SalePrice'], color='#aec7e8', ax=axes[1])
axes[1].set_title("Boxplot Harga Rumah (Deteksi Outlier)")
axes[1].set_xlabel("SalePrice ($)")

plt.tight_layout()
plt.show()

print(f"Skewness SalePrice : {df_train['SalePrice'].skew():.2f}")
print(f"Kurtosis SalePrice : {df_train['SalePrice'].kurt():.2f}")
print(f"Harga Minimum      : ${df_train['SalePrice'].min():,}")
print(f"Harga Median       : ${df_train['SalePrice'].median():,}")
print(f"Harga Rata-rata    : ${df_train['SalePrice'].mean():,.2f}")
print(f"Harga Maksimum     : ${df_train['SalePrice'].max():,}")


**Insight Target:**
- `SalePrice` memiliki *positive skewness* (condong ke kanan) sekitar 1.88, yang berarti sebagian besar rumah berada pada rentang harga murah-menengah, sementara beberapa rumah mewah memiliki harga sangat tinggi.
- Median harga rumah berada di kisaran $163,000, sedangkan nilai rata-ratanya sekitar $180,921.


## 7. Analisis Korelasi Fitur Numerik terhadap `SalePrice`
Mencari fitur mana saja yang memiliki korelasi linear tertinggi terhadap `SalePrice` untuk dijadikan kandidat fitur model regresi.


In [ ]:
# Menghitung matriks korelasi numerik
numeric_cols = df_train.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()

# Top 15 Fitur dengan Korelasi Tertinggi terhadap SalePrice
top_corr_features = corr_matrix['SalePrice'].sort_values(ascending=False).head(15)

print("=== TOP 15 FITUR PALING BERKORELASI DENGAN SALEPRICE ===")
display(top_corr_features)


In [ ]:
# Heatmap Korelasi Top 10 Fitur
top_10_cols = top_corr_features.index[:11]
plt.figure(figsize=(10, 8))
sns.heatmap(df_train[top_10_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Heatmap Korelasi Top 10 Fitur dengan SalePrice", fontsize=14)
plt.show()


## 8. Eksplorasi Khusus: 10 Fitur Pilihan yang Digunakan Model
Model Machine Learning pada file `DecisionTreeRegressor` dan `RandomForestRegressor` memilih 10 fitur berikut:
1. `OverallQual`: Kualitas material dan penyelesaian rumah secara keseluruhan (skala 1-10)
2. `GrLivArea`: Luas area tempat tinggal di atas tanah (square feet)
3. `GarageCars`: Kapasitas muat mobil di garasi
4. `TotalBsmtSF`: Total luas lantai basement (square feet)
5. `1stFlrSF`: Luas lantai pertama (square feet)
6. `YearBuilt`: Tahun konstruksi awal
7. `YearRemodAdd`: Tahun renovasi
8. `FullBath`: Jumlah kamar mandi lengkap di atas tanah
9. `TotRmsAbvGrd`: Total jumlah ruangan di atas tanah (tidak termasuk kamar mandi)
10. `LotArea`: Luas tanah (square feet)


In [ ]:
model_features = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 
    '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'FullBath', 
    'TotRmsAbvGrd', 'LotArea'
]

print("=== CEK MISSING VALUES PADA 10 FITUR MODEL ===")
print("Train Set:")
print(df_train[model_features].isnull().sum())
print("\nTest Set:")
print(df_test[model_features].isnull().sum())


**Perhatian penting:**
Pada data test, terdapat missing value pada `GarageCars` (1 baris) dan `TotalBsmtSF` (1 baris). Itulah sebabnya pada model `RandomForestRegressor`, kita melakukan **imputasi median** agar model tidak mengalami galat saat melakukan prediksi pada data test.


In [ ]:
# Visualisasi Hubungan Top 4 Fitur terhadap SalePrice
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. OverallQual vs SalePrice
sns.boxplot(x=df_train['OverallQual'], y=df_train['SalePrice'], ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title("OverallQual vs SalePrice")

# 2. GrLivArea vs SalePrice (dengan garis regresi)
sns.regplot(x=df_train['GrLivArea'], y=df_train['SalePrice'], ax=axes[0, 1], 
            scatter_kws={'alpha': 0.5, 'color': '#2ca02c'}, line_kws={'color': 'red'})
axes[0, 1].set_title("GrLivArea vs SalePrice")

# 3. GarageCars vs SalePrice
sns.boxplot(x=df_train['GarageCars'], y=df_train['SalePrice'], ax=axes[1, 0], palette="magma")
axes[1, 0].set_title("GarageCars vs SalePrice")

# 4. TotalBsmtSF vs SalePrice
sns.regplot(x=df_train['TotalBsmtSF'], y=df_train['SalePrice'], ax=axes[1, 1], 
            scatter_kws={'alpha': 0.5, 'color': '#9467bd'}, line_kws={'color': 'red'})
axes[1, 1].set_title("TotalBsmtSF vs SalePrice")

plt.tight_layout()
plt.show()


## 9. Perbandingan Distribusi Fitur: Data Train vs Data Test
Memeriksa apakah sebaran fitur pada data latih dan data uji serupa (*distribution consistency*).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fitur_banding = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'YearBuilt', 'LotArea']

for i, col in enumerate(fitur_banding):
    ax = axes[i // 3, i % 3]
    sns.kdeplot(df_train[col], label='Train', ax=ax, color='blue', fill=True, alpha=0.3)
    sns.kdeplot(df_test[col], label='Test', ax=ax, color='orange', fill=True, alpha=0.3)
    ax.set_title(f"Distribusi {col}: Train vs Test")
    ax.legend()

plt.tight_layout()
plt.show()


**Kesimpulan Perbandingan Train vs Test:**
Distribusi fitur pada data train dan data test sangat konsisten dan berhimpitan, menandakan bahwa model yang dilatih pada data train memiliki representasi data yang cocok untuk memprediksi data test.


## 10. Deteksi Pencilan Ekstrem (Outlier Detection)
Berdasarkan dokumentasi resmi dari pembuat dataset Kaggle House Prices (Dean De Cock):
Terdapat rumah dengan luas `GrLivArea` > 4000 sq ft yang dijual dengan harga sangat murah (kemungkinan penjualan parsial / non-pasar normal).


In [ ]:
# Deteksi outlier GrLivArea > 4000
outliers = df_train[(df_train['GrLivArea'] > 4000) & (df_train['SalePrice'] < 300000)]
print("Rumah dengan GrLivArea > 4000 dan SalePrice < 300,000:")
display(outliers[['Id', 'GrLivArea', 'OverallQual', 'SalePrice']])

plt.figure(figsize=(10, 6))
sns.scatterplot(x=df_train['GrLivArea'], y=df_train['SalePrice'], alpha=0.6, label='Normal Data')
sns.scatterplot(x=outliers['GrLivArea'], y=outliers['SalePrice'], color='red', s=120, label='Potensi Outlier')
plt.title("Deteksi Outlier pada GrLivArea vs SalePrice")
plt.xlabel("GrLivArea (Luas area tinggal sq ft)")
plt.ylabel("SalePrice ($)")
plt.axvline(4000, color='red', linestyle='--', alpha=0.5)
plt.legend()
plt.show()


---
## Rangkuman Temuan EDA
1. **Dimensi**: Dataset Train memiliki 1,460 baris & 81 kolom, sedangkan Test memiliki 1,459 baris & 80 kolom.
2. **Missing Values**:
   - Kolom model `GarageCars` dan `TotalBsmtSF` memiliki 1 baris nilai kosong pada data test. Imputasi median sangat disarankan untuk mencegah error saat inferensi data test.
3. **Korelasi Terkuat**:
   - `OverallQual` (korelasi 0.79) dan `GrLivArea` (korelasi 0.71) merupakan prediktor paling signifikan terhadap harga rumah.
4. **Distribusi Target**:
   - `SalePrice` mengalami right-skewed dengan nilai rata-rata sekitar $180,921 dan nilai tengah $163,000.
5. **Konsistensi Data**:
   - Sebaran fitur pada data train dan data test sangat selaras, sehingga model yang dilatih pada data train dapat melakukan generalisasi dengan baik pada data test.
